# Processed Data Quality Assessment

## Objective

Validate quality of **cleaned and processed datasets** after pipeline execution:
- Table inventory and row counts
- Join coverage across key relationships (FK integrity)
- Missing value patterns in processed tables
- Quality profile exports for audit trail

**Run this AFTER**: Quality checks on raw data (04_data_quality.ipynb)

**Output**: Reports in `reports/tables/` for downstream validation

In [ ]:
import sys
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

## Configuration & Data Loading

In [ ]:
# Resolve project root
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    for parent in PROJECT_ROOT.parents:
        if (parent / "src").exists() and (parent / "data").exists():
            PROJECT_ROOT = parent
            break

if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from instacart_quality import load_quality_table

# Configuration: switch between sample and full dataset
USE_SAMPLE = False  # Set to True for quick testing

TABLE_NAMES = [
    "orders",
    "order_products_prior",
    "order_products_train",
    "products",
    "aisles",
    "departments",
    "customer_cumulative_reorder",
]

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"Data mode: {'SAMPLE' if USE_SAMPLE else 'FULL DATASET'}")

In [ ]:
def _safe_table_load(table_name: str) -> pd.DataFrame:
    """Load table with fallback handling for missing files."""
    try:
        return load_quality_table(table_name, use_sample=USE_SAMPLE, project_root=PROJECT_ROOT)
    except FileNotFoundError as exc:
        print(f"[WARN] {table_name} not available in selected mode: {exc}")
        return pd.DataFrame()


# Load all processed tables
all_tables = {name: _safe_table_load(name) for name in TABLE_NAMES}

loaded_count = sum(1 for df in all_tables.values() if not df.empty)
print(f"\n✅ Loaded {loaded_count}/{len(TABLE_NAMES)} processed tables")

## 1. Table Inventory

In [ ]:
def build_table_inventory(table_map: dict[str, pd.DataFrame]) -> pd.DataFrame:
    """Generate table-level inventory: rows, columns, memory, duplicates."""
    rows = []
    for name, data in table_map.items():
        if data.empty:
            rows.append(
                {
                    "table": name,
                    "rows": 0,
                    "columns": 0,
                    "memory_mb": 0.0,
                    "duplicate_rows": 0,
                    "duplicate_pct": 0.0,
                    "status": "missing_or_empty",
                }
            )
            continue

        duplicate_rows = int(data.duplicated().sum())
        rows.append(
            {
                "table": name,
                "rows": int(len(data)),
                "columns": int(data.shape[1]),
                "memory_mb": round(float(data.memory_usage(deep=True).sum()) / (1024**2), 2),
                "duplicate_rows": duplicate_rows,
                "duplicate_pct": round(duplicate_rows / len(data) * 100, 3) if len(data) else 0.0,
                "status": "loaded",
            }
        )

    return pd.DataFrame(rows).sort_values("table").reset_index(drop=True)


inventory_df = build_table_inventory(all_tables)
print("📊 TABLE INVENTORY\n")
display(inventory_df)

## 2. Join Coverage (Foreign Key Integrity)

In [ ]:
def join_coverage(left: pd.DataFrame, right: pd.DataFrame, key: str, left_name: str, right_name: str) -> dict:
    """Check how many rows in left table match keys in right table (FK validation)."""
    if left.empty or right.empty:
        return {
            "left_table": left_name,
            "right_table": right_name,
            "join_key": key,
            "left_rows": len(left),
            "matched_rows": 0,
            "unmatched_rows": len(left),
            "match_rate_pct": 0.0,
            "note": "one_or_both_tables_missing",
        }

    right_keys = right[[key]].drop_duplicates()
    probe = left[[key]].merge(right_keys, on=key, how="left", indicator=True)
    matched_rows = int((probe["_merge"] == "both").sum())
    unmatched_rows = int((probe["_merge"] == "left_only").sum())

    return {
        "left_table": left_name,
        "right_table": right_name,
        "join_key": key,
        "left_rows": int(len(left)),
        "matched_rows": matched_rows,
        "unmatched_rows": unmatched_rows,
        "match_rate_pct": round(matched_rows / len(left) * 100, 3) if len(left) else 0.0,
        "note": "✅ ok" if unmatched_rows == 0 else "⚠️ ORPHAN RECORDS FOUND",
    }


join_quality_df = pd.DataFrame(
    [
        join_coverage(all_tables["order_products_prior"], all_tables["orders"], "order_id", "order_products_prior", "orders"),
        join_coverage(all_tables["order_products_train"], all_tables["orders"], "order_id", "order_products_train", "orders"),
        join_coverage(all_tables["order_products_prior"], all_tables["products"], "product_id", "order_products_prior", "products"),
        join_coverage(all_tables["order_products_train"], all_tables["products"], "product_id", "order_products_train", "products"),
        join_coverage(all_tables["products"], all_tables["aisles"], "aisle_id", "products", "aisles"),
        join_coverage(all_tables["products"], all_tables["departments"], "department_id", "products", "departments"),
    ]
)

print("\n🔗 JOIN COVERAGE CHECKS (Foreign Key Integrity)\n")
display(join_quality_df)

## 3. Missing Values in Processed Tables

In [ ]:
missing_by_column = (
    pd.concat(
        [
            pd.DataFrame({
                "table": name,
                "column": df.columns,
                "missing_count": df.isna().sum().values,
                "missing_pct": (df.isna().mean() * 100).values,
            })
            for name, df in all_tables.items() if not df.empty
        ],
        ignore_index=True,
    )
    .query('missing_count > 0')
    .sort_values(['missing_pct', 'missing_count'], ascending=False)
    .reset_index(drop=True)
)

if missing_by_column.empty:
    print("\n✅ NO MISSING VALUES DETECTED in processed tables")
else:
    print("\n⚠️ MISSING VALUES BY COLUMN\n")
    display(missing_by_column)

## 4. Export Quality Reports

In [ ]:
# Create reports directory
mode_tag = "sample" if USE_SAMPLE else "full"
reports_table_dir = PROJECT_ROOT / "reports" / "tables"
reports_table_dir.mkdir(parents=True, exist_ok=True)

# Export processed data quality files
inventory_path = reports_table_dir / f"processed_data_inventory_{mode_tag}.csv"
join_path = reports_table_dir / f"processed_join_quality_{mode_tag}.csv"
missing_path = reports_table_dir / f"processed_missing_values_{mode_tag}.csv"

inventory_df.to_csv(inventory_path, index=False)
join_quality_df.to_csv(join_path, index=False)
if not missing_by_column.empty:
    missing_by_column.to_csv(missing_path, index=False)

print("\n✅ EXPORTED QUALITY REPORTS to reports/tables/")
print(f"   📄 processed_data_inventory_{mode_tag}.csv")
print(f"   📄 processed_join_quality_{mode_tag}.csv")
if not missing_by_column.empty:
    print(f"   📄 processed_missing_values_{mode_tag}.csv")

print(f"\n✅ QUALITY ASSESSMENT COMPLETE")
print(f"   Tables checked: {loaded_count}/{len(TABLE_NAMES)}")
print(f"   Total rows scanned: {inventory_df['rows'].sum():,}")
print(f"   Join coverage status: {len(join_quality_df[join_quality_df['note'] == '✅ ok'])} / {len(join_quality_df)} relationships OK")